In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

In [3]:
file_path = "/content/drive/MyDrive/ML Challenge Dataset/TRAIN.csv"

df = pd.read_csv(file_path)

In [12]:
df = df.drop_duplicates().reset_index(drop=True)
# with duplicates auc was 0.998

In [13]:
features = [c for c in df.columns if c != "Class"]

df["row_mean"] = df[features].mean(axis=1)
df["row_std"] = df[features].std(axis=1)
df["row_max"] = df[features].max(axis=1)
df["row_min"] = df[features].min(axis=1)

In [14]:
skewed = ["F30","F38","F37","F36","F35","F34","F39"]

for col in skewed:
    df[col] = np.log1p(df[col] - df[col].min() + 1)

In [15]:
X = df.drop("Class",axis=1)
y = df["Class"]

In [16]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = []

In [17]:
for fold,(train_idx,val_idx) in enumerate(folds.split(X,y)):

    X_train,X_val = X.iloc[train_idx],X.iloc[val_idx]
    y_train,y_val = y.iloc[train_idx],y.iloc[val_idx]

    # LightGBM
    lgb_model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    # XGBoost
    xgb_model = XGBClassifier(
        n_estimators=800,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )

    # Random Forest
    rf_model = RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    )

    # Train
    lgb_model.fit(X_train,y_train)
    xgb_model.fit(X_train,y_train)
    rf_model.fit(X_train,y_train)

    # Predict probabilities
    lgb_pred = lgb_model.predict_proba(X_val)[:,1]
    xgb_pred = xgb_model.predict_proba(X_val)[:,1]
    rf_pred = rf_model.predict_proba(X_val)[:,1]

    # Ensemble (Kaggle trick)
    val_pred = 0.4*lgb_pred + 0.4*xgb_pred + 0.2*rf_pred

    oof_preds[val_idx] = val_pred

    score = roc_auc_score(y_val,val_pred)

    print("Fold",fold,"AUC:",score)

[LightGBM] [Info] Number of positive: 13849, number of negative: 20581
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020514 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13005
[LightGBM] [Info] Number of data points in the train set: 34430, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.402236 -> initscore=-0.396155
[LightGBM] [Info] Start training from score -0.396155
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

In [18]:
print("Final CV AUC:",roc_auc_score(y,oof_preds))

Final CV AUC: 0.9984090402243647


In [19]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import lightgbm as lgb

y_shuffled = y.sample(frac=1, random_state=42).reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for train_idx, val_idx in skf.split(X, y_shuffled):

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_shuffled.iloc[train_idx], y_shuffled.iloc[val_idx]

    model = lgb.LGBMClassifier(n_estimators=500)

    model.fit(X_train, y_train)

    preds = model.predict_proba(X_val)[:,1]

    score = roc_auc_score(y_val, preds)

    scores.append(score)

print("Mean AUC with shuffled labels:", np.mean(scores))

[LightGBM] [Info] Number of positive: 13849, number of negative: 20581
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.045932 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13005
[LightGBM] [Info] Number of data points in the train set: 34430, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.402236 -> initscore=-0.396155
[LightGBM] [Info] Start training from score -0.396155
[LightGBM] [Info] Number of positive: 13849, number of negative: 20581
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023979 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13005
[LightGBM] [Info] Number of data points in the train set: 34430, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.402236 -> initscore=-0.396155
[LightGBM] [Info] Start training from score -0.396155
[LightGBM]

In [21]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import pandas as pd
import numpy as np

results = {}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for col in X.columns:

    scores = []

    for train_idx, val_idx in skf.split(X, y):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(
            n_estimators=200,
            verbose=-1
        )

        model.fit(X_train[[col]], y_train)

        preds = model.predict_proba(X_val[[col]])[:,1]

        scores.append(roc_auc_score(y_val, preds))

    results[col] = np.mean(scores)

# show top features
pd.Series(results).sort_values(ascending=False).head(15)

,0
F29,0.755704
F01,0.754529
F09,0.747783
F26,0.735942
F19,0.729925
F06,0.727068
F21,0.719986
F28,0.718085
F07,0.715757
F08,0.714557
